---
title: "DRG Grouping"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

In [1]:
system("git submodule update --init --recursive")
# system("git submodule foreach --recursive git fetch && git submodule foreach --recursive && git reset --hard origin/main")
Sys.setenv(PYTHONPATH = here::here("data-cleaning", "grouper"))


In [2]:
# Delete all R objects and run garbage collection so we start with a clean slate
rm(list = ls())
gc()


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,657009,35.1,1444480,77.2,1369814,73.2
Vcells,1219194,9.4,8388608,64.0,1924961,14.7


In [3]:
# Load libraries and minor parameters
source(here::here("data-cleaning/r_scripts", "00_libraries-params.R"))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc_gmail_com/drg-pipeline

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: docstring


Attaching package: ‘docstring’


The following object is masked from ‘package:utils’:

    ?


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The foll

In [4]:
# List of Python packages to install
pkgs <- c("numpy", "pandas", "streamlit", "python_dateutil", "tabulate", "swifter", "rpy2", "pyreadr", "re")

# Install Python packages for reticulate only if they are not already installed
for (pkg in pkgs) if (!py_module_available(pkg)) py_install(pkg)


Using virtual environment '/home/resurreccion_cmc_gmail_com/.virtualenvs/r-reticulate' ...


+ /home/resurreccion_cmc_gmail_com/.virtualenvs/r-reticulate/bin/python -m pip install --upgrade --no-user python_dateutil



## Primary Parameters

In [5]:
# Prompt Options:
to_prompt <- FALSE # Whether to prompt for user inputs or not (if FALSE, default values in this cell will be used)
thai_prompt <- TRUE # Whether to prompt for thai grouper even if bypassing all other prompts

# IMPORTANT PARAMETERS:
full_claims_prefix <- "claims_extract_CLAIMS " # Include spaces if there are any
# Assign the correct file extension based on the year
# Read the contents of year_to_load.txt as a string
year_to_load <- fread(here::here("data-cleaning", "cache", "year_to_load.txt"), header = FALSE, colClasses = "character")[[1]]
print(year_to_load)

# MANUAL OVERRIDE
year_to_load <- "2019"

file_type <- if (year_to_load %in% c(2022:2023)) ".tsv" else ".csv"
print(file_type)
gcs_email <- "271591364028-compute@developer.gserviceaccount.com" # Service Account to use
gcp_proj <- system("gcloud config get-value project", intern = TRUE) # get current GCP Project
gcs_bucket <- "phic-claims-checkpoints" # Name of GCS bucket
gcs_pre_fpath <- "pre-tdrg" # Name of folder path prefix in GCS bucket for thai grouper input
gcs_post_fpath <- "post-tdrg" # Name of folder path prefix in GCS bucket for thai grouper output
gcs_spc_fpath <- "spc"
bq_dataset <- "phic" # bq dataset
bq_table <- paste0("temp_claims_", year_to_load) # temp bq table, later renamed to claims_20XX1231 in Push to BQ section

# Input:
to_spc <- TRUE
to_sample <- TRUE # Whether to sample each split_part by sample_size_divisor (useful when iterating through code runs in quick succession)
sample_size_divisor <- 25 # Sample size divisor: Formula for sample size is total_rows / split_parts / sample_size_divisor. Choose between 5, 25, 125, and 625

# Output:
to_write <- TRUE # Whether to write out checkpoint_1 files (everything up until converting for grouper export)
to_combine <- TRUE # Whether to combine checkpoint 1 files into one data.table
to_group <- TRUE # Whether to export for the batch grouper or not
to_gcs <- TRUE # Whether to push to GCS or nt (Thai Grouper Input/Output)
to_bq <- TRUE # Whether to push to BQ or not
to_drop_bq <- TRUE # Whether to overwrite the existing BQ table

# Columns to drop
drop_cols <- c( # Which columns to drop
  paste0("ICDCODE", 13:14), # Start
  "ICCODED15", # note that ICDCODE15 is misspelled as ICCODED15 in all claims
  paste0("ICDCODE", 16:170) # Continuation
)

drop_cols_manual <- c(
  "MEMCAT_SUBCHILD_DESC" # Drop as per Cel's suggestion
)

# Flush files
to_flush_master <- FALSE # whether to flush aux-files, checkpoints, profvis, debug, cache, and samples
to_flush_partial <- FALSE # whether to flush partial files (raw files but split into split_parts parts)

# Control random behavior for reproducibility
global_seed <- seed <- 123 # Choose a number as seed
set.seed(seed) # Setting the seed reproducibility (Important for stuff like randomly choosing a pdx among multiple possible options)

# Machine Specifications
ram_size <- 64 # Input virtual or physical machine's RAM size here

# Print GCP project
message(paste("GCP Project:", gcp_proj, "\n"))


[1] "2022"
[1] ".csv"


GCP Project: drg-pipeline 




In [6]:
# other parameters for manual adjustments
manual_patterns_to_replace <- c("\\b0800\\b", "\\b080\\b", "\\b0809\\b") # ICD codes to replace
manual_code_replacements <- c("O800", "O80", "O809") # ICD code replacements


## Secondary (Debug) Parameters

In [7]:
# Debug Parameters:
to_debug <- FALSE # whether to print debug statements
to_profvis <- FALSE # Conduct runtime duration analysis via profvis or not
to_view_checks <- TRUE # Whether to view checks and print statements
to_view_checks_parallel <- FALSE # Whether to view intermediate per split_part/chunk checks and print statements (not consolidated) when parallelized
to_parallel <- TRUE # Whether to parallelize each split_parts split_part into availableCores() chunks. Cuts down processing time from 120min to 15min.
to_split_read <- FALSE # WARNING: TRUE uses a lot of memory!!
to_dec_mem_usage <- TRUE # Whether to run rm() and gc() at every possible step
tmp_nrow <- Inf # Per split_part/chunk end_nrow (leave at Inf)
diff_chars <- 0
split_parts <- 15 # How many (integer) parts to split the 12+m row claims file into
end_nrow <- 10 # How many rows/entries to show in summary tables
encode <- "unknown" # Choices: unknown, UTF-8, Latin-1
# sep <- "," # Choices: "," or "\t"
is_unix <- if (.Platform$OS.type == "unix") TRUE else FALSE # Detect operating system architecture

ram_buffer <- 0.1 # How much of a RAM buffer to leave for the OS


## File Paths

In [8]:
# Folder Path Prefixes:
clean_prefix <- "data-cleaning"
data_prefix <- file.path(clean_prefix, "data")
claims_prefix <- file.path(data_prefix, "claims")
checkpoint_1_prefix <- "checkpoint_1_claims_"
checkpoint_2_prefix <- "checkpoint_2_claims_"
checkpoint_3_prefix <- "DRG_Grouped_"
checkpoint_4_prefix <- "checkpoint_4_thai_grouper_input_"
checkpoint_5_prefix <- toupper(paste0(gcs_pre_fpath, "_", checkpoint_4_prefix))
checkpoint_6_prefix <- "checkpoint_6_grouped_claims"
checkpoint_7a_prefix <- "python_input_1"
checkpoint_7b_prefix <- "python_input_2"
checkpoint_8_prefix <- "python_output"
checkpoint_10_prefix <- "stata"

# Folder Paths:
chkpt_path <- file.path(data_prefix, "checkpoints")
checkpoint_1_path <- file.path(chkpt_path, "checkpoint_1_partial_clean_claims")
checkpoint_2_path <- file.path(chkpt_path, "checkpoint_2_master_clean_claims")
checkpoint_3_path <- file.path(chkpt_path, "checkpoint_3_thai_partial_input")
checkpoint_4_path <- file.path(chkpt_path, "checkpoint_4_thai_master_input")
checkpoint_5_path <- file.path(chkpt_path, "checkpoint_5_thai_output")
checkpoint_6_path <- file.path(chkpt_path, "checkpoint_6_thai_merged")
checkpoint_7_path <- file.path(chkpt_path, "checkpoint_7_py_input")
checkpoint_8_path <- file.path(chkpt_path, "checkpoint_8_py_output")
checkpoint_9_path <- file.path(chkpt_path, "checkpoint_9_grouper_differences")
checkpoint_10_path <- file.path(chkpt_path, "checkpoint_10_stata")
cache_path <- file.path(clean_prefix, "cache")
aux_path <- file.path(data_prefix, "aux-files")
raw_claims_path <- file.path(claims_prefix, "raw")
raw_claims_parts_path <- file.path(claims_prefix, "raw", "parts")
raw_claims_samples_path <- file.path(claims_prefix, "raw", "samples")
profvis_path <- file.path(data_prefix, "profvis")
debug_path <- file.path("data-cleaning", "debug")

# File Paths
profvis_fpath <- here("data-cleaning", "data", "profvis", "profvis.html")

# Create directories:
created_dirs <- c() # Initialize empty vector
# For all "_path" variables, create a directory with that path
# Excludes "_fpath" variables
for (path in mget(ls(pattern = "_path$"), envir = .GlobalEnv)) {
  full_path <- here(path)
  if (!dir.exists(full_path)) {
    dir.create(full_path, recursive = TRUE)
    created_dirs <- c(created_dirs, full_path)
  }
}

# Print directories created if any
if (length(created_dirs) == 0) {
  message("All directories exist.\n")
} else {
  message("The following directories were created:\n")
  message(paste(paste(created_dirs, collapse = ",\n"), "\n"))
}

# Commonly Used File Paths:
full_claims_file <- here(
  raw_claims_path,
  paste0(full_claims_prefix, year_to_load, file_type) # Use the file_type variable here
)


All directories exist.




## Parameter Validation Logic

Checking if parameters are valid, especially for the current machine type (e.g. RAM size)

Additionally, ask for parameters if to_bypass_prompts is false

In [9]:
# Stop if forecasted memory usage is expected to crash the system
if (!split_parts == as.integer(split_parts) || split_parts <= 1) stop("ERROR: split_parts must be an integer greater than or equal to 2!")
if (ram_size <= 64 && split_parts <= 2) stop("Please set split_parts to at least 3 for 64 GB machines or it will likely crash")
if (ram_size <= 32 && split_parts <= 4) stop("Please set split_parts to at least 5 for 32 GB machines or it will likely crash")
if (ram_size <= 32 && to_split_read == TRUE) stop("Please set to_split_read to TRUE for 32 GB machines or it will likely crash")

# Function to prompt for input with default value
prompt_with_default <- function(prompt_text, default_value) {
  if (to_prompt) {
    user_input <- readline(prompt = paste0(prompt_text, " [Default: ", default_value, "]: "))
    if (user_input == "") {
      return(default_value)
    } else {
      return(user_input)
    }
  } else {
    message(
      paste0(
        "Using default value (",
        default_value, ") for ",
        deparse(substitute(default_value))
      )
    )
    return(default_value)
  }
}

# Set parameters based on prompts or defaults
full_claims_prefix <- prompt_with_default("Enter full_claims_prefix", full_claims_prefix)
full_claims_bq_prefix <- str_replace_all(full_claims_prefix, " ", "\\\\ ")
year_to_load <- prompt_with_default("Enter year_to_load", year_to_load)

# Prompt for whether to sample
to_sample <- as.logical(prompt_with_default("Sample data? (TRUE/FALSE)", to_sample))
sample_size_divisor <- as.integer(prompt_with_default("Enter sample_size_divisor", sample_size_divisor))

# Prompt for output options
to_write <- as.logical(prompt_with_default("Write output files? (TRUE/FALSE)", to_write))
to_combine <- as.logical(prompt_with_default("Combine files? (TRUE/FALSE)", to_combine))
to_group <- as.logical(prompt_with_default("Export for batch grouper? (TRUE/FALSE)", to_group))

# Flush options
to_flush_master <- as.logical(prompt_with_default("Flush master files? (TRUE/FALSE)", to_flush_master))
to_flush_partial <- as.logical(prompt_with_default("Flush partial files? (TRUE/FALSE)", to_flush_partial))

# RAM settings
ram_size <- as.numeric(prompt_with_default("Enter RAM size (GB)", ram_size))
# ram_buffer <- as.numeric(prompt_with_default("Enter RAM buffer (0.0-1.0)", ram_buffer))
ram_limit <- (1 - ram_buffer) * ram_size * (1024^3)

# Allowing each future_lapply session to use more memory
options(future.globals.maxSize = ram_limit)

# Compute the RAM limit for R processes, leaving the buffer for the OS
ram_limit_gb <- round((1 - ram_buffer) * ram_size, 0)

# Print the set RAM limit
message(sprintf("Setting future.globals.maxSize to: %.1f GB", ram_limit / (1024^3)))


Using default value (claims_extract_CLAIMS ) for full_claims_prefix

Using default value (2019) for year_to_load

Using default value (TRUE) for to_sample

Using default value (25) for sample_size_divisor

Using default value (TRUE) for to_write

Using default value (TRUE) for to_combine

Using default value (TRUE) for to_group

Using default value (FALSE) for to_flush_master

Using default value (FALSE) for to_flush_partial

Using default value (64) for ram_size

Setting future.globals.maxSize to: 57.6 GB



## Loading


### Load Required Libraries & Initial Functions

In [10]:
# Hide verbose outputs and warnings
options(verbose = FALSE) # Hide verbose output for script and library loading
options(warn = -1) # Hide warnings for script sourcing and library loading


In [11]:
scripts <- list( # List of scripts to source
  # lib_params = "00_libraries-params.R",
  cleaning = "01_cleaning-functions.R",
  clinical = "02_clinical-functions.R",
  timing_debug = "03_timing-debug-functions.R",
  summary = "04_summary-functions.R",
  io = "05_io-functions.R"
)

# Loop to source above scripts
for (script in scripts) source(here("data-cleaning/r_scripts", script))

# Load cached total rows file if available, saves ~10 seconds of runtime
total_rows_file <- here(cache_path, paste0("total_rows_", year_to_load, ".rds"))
if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
  message(paste("Total Rows via cached object:", total_rows))
} else {
  total_rows <- fread(file = full_claims_file, select = 1L, header = TRUE, colClasses = "character")[, .N]
  saveRDS(total_rows, file = total_rows_file)
  message(paste("Total Rows via fread:", total_rows))
}

# Compute sample size when splitting and when not,
# only relevant when sampling
if (to_split) {
  sample_size <- ceiling(total_rows / split_parts / sample_size_divisor)
} else {
  sample_size <- ceiling(total_rows / sample_size_divisor)
}

# suffix appended to files to indicate if they are from sampled or full runs
suffix <- paste0(
  ifelse(to_sample, paste0("_sampled_", sample_size, "_"), "_full_")
)


Total Rows via cached object: 12562622



In [12]:
# since verbose = TRUE is way too verbose compared to default
options(warn = 1) # Reenable warnings; see above comments


In [ ]:
# Load the data
result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))

# Subset the rows where pat_bdate_recomputed is generated and assign them to pat_bdate_recomputed
pat_bdate_recomputed <<- result[
  # !is.na(pat_age) & is.na(pat_bdate)
  ,
  .(id_series, pat_bdate_recomputed = dmy(generate_dob(format(pat_bdate, "%Y-%m-%d"), pat_age, format(date_adm, "%Y-%m-%d"))))
]

str(pat_bdate_recomputed)

result[!is.na(pat_bdate) & pat_bdate <= date_adm & floor(as.numeric(interval(pat_bdate, date_adm) / years(1))) >= 0, pat_age := floor(as.numeric(interval(pat_bdate, date_adm) / years(1)))]

result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 & is.na(pat_ageday), pat_ageday := 3]

result[
  !is.na(pat_age) & pat_age >= 0 & pat_age < 1 & !is.na(date_adm) & !is.na(pat_bdate),
  pat_ageday := as.integer(difftime(date_adm, pat_bdate, units = "days"))
]

if ("ageday" %in% colnames(result)) {
  result[, ageday := NULL]
}
gc()
# # Print messages for invalid ageday corrections
# invalid_ageday_after <- result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 & is.na(pat_ageday), .N]
# message(
#   "Number of agedays generated: ", invalid_ageday_before - invalid_ageday_after,
#   ". Agedays generated are for where the patient is younger than 1 year, so the exact number of days was generated."
# )

bw_dist <- c(
  runif(2, 0.5, 0.9), # Random birthweight between 0.5 and 0.9 for 2 newborns
  runif(8, 1.1, 1.4), # Random birthweight between 1.1 and 1.4 for 8 newborns
  runif(19, 1.6, 1.9), # Random birthweight between 1.6 and 1.9 for 19 newborns
  runif(95, 2.1, 2.4), # Random birthweight between 2.1 and 2.4 for 95 newborns
  runif(381, 2.6, 2.9), # Random birthweight between 2.6 and 2.9 for 381 newborns
  runif(375, 3.1, 3.4), # Random birthweight between 3.1 and 3.4 for 375 newborns
  runif(115, 3.5, 4.0), # Random birthweight between 3.5 and 4.0 for 115 newborns
  runif(6, 0.5, 4.0) # Random birthweight between 0.5 and 4.0 for 6 newborns
)

# Create the zero_mask condition where pat_age is between 0 and 1 (newborns)
zero_mask <- result[, pat_age >= 0 & pat_age < 1]

# Apply birthweight only if pat_bwt is NA and zero_mask is TRUE
result[(pat_bwt < 0 | is.na(pat_bwt)) & zero_mask, pat_bwt := sapply(.SD$pat_bwt, function(x) sample(bw_dist, 1)), .SDcols = "pat_bwt"]


## Run Python Grouper

Prepare data for python grouper

In [93]:
# Convert relevant data types
result[, patage := as.numeric(pat_age)]
result[, patsex := as.character(pat_sex)]
result[, birthweight := as.numeric(pat_bwt)]
result[, discharge := as.integer(clin_discharge)]
result[, ageday := as.integer(pat_ageday)]
result[, pdx := clin_pdx]

# Handle ICD splitting (clin_icd to sdx columns)
split_codes <- function(dt, column, prefix, max_cols) {
  split_list <- lapply(dt[[column]], function(x) unlist(strsplit(x, "\\|")))
  split_cols <- lapply(1:max_cols, function(i) sapply(split_list, function(x) if (length(x) >= i) x[[i]] else NA_character_))
  split_dt <- as.data.table(split_cols)
  setnames(split_dt, paste0(prefix, 1:max_cols))
  return(split_dt)
}

sdx_columns <- split_codes(result, "clin_sdx", "sdx", 12)
proc_columns <- split_codes(result, "clin_proc", "proc", 20)

# Combine the split columns back into result
result <- cbind(result, sdx_columns, proc_columns)

# Replace NA in non-date columns with "None"
non_date_columns <- c("patsex", "pdx", paste0("sdx", 1:12), paste0("proc", 1:20))
result[, (non_date_columns) := lapply(.SD, function(x) ifelse(is.na(x), "None", x)), .SDcols = non_date_columns]

# Prepare the final data table for writing
for_fwrite <- result[, c(
  "id_series", "date_adm", "date_dis", "time_adm", "time_dis", "patage", "patsex", "discharge", "pdx",
  paste0("sdx", 1:12), paste0("proc", 1:20), "birthweight", "ageday"
), with = FALSE]

for_fwrite[, date_adm := as.character(paste(date_adm, time_adm), format = "%Y-%m-%d %H:%M:%S")]
for_fwrite[, date_dis := as.character(paste(date_dis, time_dis), format = "%Y-%m-%d %H:%M:%S")]

for_fwrite[, time_adm := NULL]
for_fwrite[, time_dis := NULL]

# Write the final table to a CSV file
fwrite(for_fwrite, here(checkpoint_7_path, paste0(checkpoint_7b_prefix, suffix, ".csv")))

# Create a summary table that shows the count of non-null values for each column
summary_table <- for_fwrite[, lapply(.SD, function(x) sum(!is.na(x) & x != "None")), .SDcols = names(for_fwrite)]

# Transpose the summary table to make it more readable
summary_table <- transpose(summary_table)
setnames(summary_table, "Non-Null Count")
summary_table[, Column := names(for_fwrite)]

# Reorder the summary table to show the columns
setcolorder(summary_table, c("Column", "Non-Null Count"))

# Print the summary table
print(summary_table)


         Column Non-Null Count
         <char>          <int>
 1:   id_series         502515
 2:    date_adm         502515
 3:    date_dis         502515
 4:      patage         502366
 5:      patsex         502515
 6:   discharge         502274
 7:         pdx         465832
 8:        sdx1         204822
 9:        sdx2          91356
10:        sdx3          32332
11:        sdx4          11638
12:        sdx5           5083
13:        sdx6           2444
14:        sdx7           1222
15:        sdx8            644
16:        sdx9            357
17:       sdx10            200
18:       sdx11            107
19:       sdx12             37
20:       proc1         160021
21:       proc2           3400
22:       proc3             83
23:       proc4             10
24:       proc5              1
25:       proc6              0
26:       proc7              0
27:       proc8              0
28:       proc9              0
29:      proc10              0
30:      proc11              0
31:     

In [94]:
# csv_path <- here(checkpoint_7_path, paste0(checkpoint_7b_prefix, suffix, ".rds"))

# Load Python's pandas via reticulate
pandas <- import("pandas")

# Assuming 'test' is your R data.frame or data.table, convert it to a pandas DataFrame
py$pandas_df <- pandas$DataFrame(as.data.frame(for_fwrite))

py_run_string("
import pandas as pd
import numpy as np
from io import StringIO
import sys

# Capture the output in a string buffer
old_stdout = sys.stdout
sys.stdout = mystdout = StringIO()

# Convert the column types explicitly
pandas_df['patage'] = pd.to_numeric(pandas_df['patage'], errors='coerce')
pandas_df['birthweight'] = pd.to_numeric(pandas_df['birthweight'], errors='coerce')
pandas_df['discharge'] = pandas_df['discharge'].astype('Int64')

# Convert string columns to 'string' dtype and replace NA values with None
string_columns = ['id_series', 'patsex', 'pdx', 'sdx1', 'sdx2', 'sdx3', 'sdx4', 'sdx5', 'sdx6', 'sdx7', 'sdx8', 'sdx9', 'sdx10', 'sdx11', 'sdx12',
                  'proc1', 'proc2', 'proc3', 'proc4', 'proc5', 'proc6', 'proc7', 'proc8', 'proc9', 'proc10', 'proc11', 'proc12',
                  'proc13', 'proc14', 'proc15', 'proc16', 'proc17', 'proc18', 'proc19', 'proc20', 'date_adm', 'date_dis']

# Replace NA, pd.NA, '<NA>', 'None', 'NA' in string columns with None
pandas_df[string_columns] = pandas_df[string_columns].replace([pd.NA, np.nan, '<NA>', 'None', 'NA'], 'None')

pandas_df = pandas_df.replace(pd.NA, None)
pandas_df = pandas_df.replace(np.nan, None)
pandas_df = pandas_df.replace('<NA>', None)
pandas_df = pandas_df.replace('None', None)
pandas_df = pandas_df.replace('NA', None)
pandas_df = pandas_df.replace(-2147483648, None)

# Convert columns to string dtype after replacing the values
pandas_df[string_columns] = pandas_df[string_columns].astype('string')

# Replace -2147483648 with None in numeric columns, if necessary
pandas_df = pandas_df.replace(-2147483648, None)

# Filter rows where 'discharge' is not in [1, 2, 3, 4, 9]
not_in_list_values = pandas_df.loc[~pandas_df['discharge'].isin([1, 2, 3, 4, 9]), 'discharge']

# Get unique values and their counts
unique_not_in_list_values = not_in_list_values.value_counts()

# Print the unique values and their counts
print(unique_not_in_list_values)

# Reset stdout and capture the output
sys.stdout = old_stdout
output = mystdout.getvalue()

# Capture pandas_df.info() output
buffer = StringIO()
pandas_df.info(buf=buffer)
info_output = buffer.getvalue()

# Sample 10,000 rows from the DataFrame
# pandas_df = pandas_df.sample(n=10000, random_state=42)
")

# Print the captured output in R
cat(py$info_output)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502515 entries, 0 to 502514
Data columns (total 41 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   id_series    502515 non-null  string
 1   date_adm     502515 non-null  string
 2   date_dis     502515 non-null  string
 3   patage       502366 non-null  object
 4   patsex       502515 non-null  string
 5   discharge    502274 non-null  Int64 
 6   pdx          465832 non-null  string
 7   sdx1         204822 non-null  string
 8   sdx2         91356 non-null   string
 9   sdx3         32332 non-null   string
 10  sdx4         11638 non-null   string
 11  sdx5         5083 non-null    string
 12  sdx6         2444 non-null    string
 13  sdx7         1222 non-null    string
 14  sdx8         644 non-null     string
 15  sdx9         357 non-null     string
 16  sdx10        200 non-null     string
 17  sdx11        107 non-null     string
 18  sdx12        37 non-null      string
 19  pr

In [95]:
# Recalculate summary_table, considering 'None' as null
summary_table <- for_fwrite[, lapply(.SD, function(x) sum(!is.na(x) & x != "None")), .SDcols = names(for_fwrite)]
summary_table <- transpose(summary_table)
setnames(summary_table, "NonNullCount_R")
summary_table[, Column := names(for_fwrite)]
setcolorder(summary_table, c("Column", "NonNullCount_R"))

# Parse info_output from Python
lines <- strsplit(py$info_output, "\n")[[1]]

# Find the indices of the data lines
start_idx <- which(grepl("^---", lines))
end_idx <- which(grepl("^dtypes:", lines)) - 1

data_lines <- lines[(start_idx + 1):end_idx]
data_lines <- data_lines[nchar(data_lines) > 0]

# Parse each line to extract the column name and non-null count
parsed_lines <- str_match(data_lines, "^\\s*(\\d+)\\s+(\\S+)\\s+(\\d+)\\s+non-null\\s+(\\S+)")

# Create python_summary data frame
python_summary <- data.table(
  Column = parsed_lines[, 3],
  NonNullCount_Python = as.integer(parsed_lines[, 4])
)

# Merge the two summaries
comparison <- merge(summary_table, python_summary, by = "Column", all = TRUE)

# Calculate the difference
comparison[, Difference := NonNullCount_R - NonNullCount_Python]

# # Print the comparison
# print(comparison)

# Identify columns with differences
differences <- comparison[Difference != 0]
if (nrow(differences) == 0) {
  cat("All non-null counts match between R and Python.\n")
} else {
  cat("Differences found in the following columns:\n")
  print(differences)
}


All non-null counts match between R and Python.


In [96]:
# Set pandas to display all rows and columns where 'patage' is NaN, then print everything using StringIO
py_run_string("
import sys
from io import StringIO
import pandas as pd

# Set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# Capture print output using StringIO
output = StringIO()
sys.stdout = output

# Filter rows where 'patage' is NaN and print the filtered DataFrame
filtered_df = pandas_df[pandas_df['patage'].isna()]
print(filtered_df)

# Reset stdout
sys.stdout = sys.__stdout__

# Get the printed output
printed_output = output.getvalue()

# Reset pandas display options to default
pd.reset_option('display.max_columns')
")

# Print the captured output in R
cat(py$printed_output)


            id_series                      date_adm  \
14937   0000004758784  2019-01-15 09:45:00 09:45:00   
24581   0000004336739  2019-01-10 13:55:00 13:55:00   
29462   0000006047006  2019-01-01 15:25:00 15:25:00   
35645   0000037613232  2019-01-25 17:58:00 17:58:00   
60075   0000048531481  2019-01-21 21:40:00 21:40:00   
...               ...                           ...   
495808  0000001919465  2019-11-29 08:00:00 08:00:00   
497365  0000032246398  2019-11-23 02:20:00 02:20:00   
499373  0000006504260  2019-12-19 13:00:00 13:00:00   
500033  0000002155680  2019-12-11 01:55:00 01:55:00   
501902  0000042888201  2019-11-25 00:06:00 00:06:00   

                            date_dis patage patsex  discharge   pdx  sdx1  \
14937   2019-01-19 11:50:00 11:50:00   None      F          1  K291  <NA>   
24581   2019-01-13 13:00:00 13:00:00   None      F          1  <NA>  <NA>   
29462   2019-01-10 15:30:00 15:30:00   None      F          1  Z380  <NA>   
35645   2019-01-28 17:45:00 17:

In [76]:
# Sys.setenv(PYTHONPATH = here("data-cleaning", "grouper"))

# Step 6: Process each row of the DataFrame through `drg_seeker` and append results
# if (!file.exists(here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")))) {
#   py_run_file(here("data-cleaning", "py_scripts", "run_drg_seeker.py"))
#   output_dt <- as.data.table(py$output)
#   saveRDS(output_dt, here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")), compress = TRUE)
#   cat(py$statements)
# } else {
#   output_dt <- readRDS(here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")))
# }
py_run_file(here("data-cleaning", "py_scripts", "run_drg_seeker.py"))
output_dt <- as.data.table(py$output)
saveRDS(output_dt, here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")), compress = TRUE)
cat(py$statements)


### Process python output to match BQ schema

In [77]:
# Assume 'pandas_df' is your DataFrame in Python after running 'run_drg_seeker.py'
# And 'output' is the result from your Python script
# Retrieve 'output' DataFrame from Python

# Rename columns to match required names if necessary
setnames(output_dt,
  old = c("drg", "pdc", "pccl", "error_code", "warning_code"),
  new = c("py_drg", "py_pdc", "py_pccl", "py_err", "py_warn"), skip_absent = TRUE
)

# Select only the required columns
required_columns <- c("id_series", "py_drg", "py_pdc", "py_pccl", "py_err", "py_warn")
output_dt <- output_dt[, ..required_columns]

# Define the format_id function
format_id <- function(x) {
  x <- as.character(x)
  integer_x <- suppressWarnings(as.integer(x))
  x <- trimws(formatC(integer_x, format = "f", digits = 0))
  x[x == "NA" | is.na(integer_x)] <- NA_character_
  x
}

# Apply format_id to 'id_series'
output_dt[, id_series := format_id(id_series)]

# Adjust data types
output_dt[, py_drg := as.character(py_drg)]
output_dt[, py_pdc := as.character(py_pdc)]
output_dt[, py_pccl := as.numeric(py_pccl)]

# Convert 'py_err' and 'py_warn' to arrays (list of character vectors)
array_columns <- c("py_err", "py_warn")

process_error_warning_column <- function(col) {
  lapply(col, function(x) {
    # Flatten x to a character vector
    x <- unlist(x)
    x <- as.character(x)

    # If x is NULL or length zero after unlisting, return character(0)
    if (is.null(x) || length(x) == 0) {
      return(character(0))
    }

    # Remove any NA values from x
    x <- x[!is.na(x)]

    # Remove any "None", "NA", or empty strings from x
    x <- x[!(x %in% c("None", "NA", "NaN", ""))]

    # If x is now length zero after cleaning, return character(0)
    if (length(x) == 0) {
      return(character(0))
    }

    # Now split each element of x by comma and optional whitespace
    split_x <- unlist(strsplit(x, ",\\s*"))

    # Remove any empty strings, "NA", or "None" from split_x
    split_x <- split_x[!(split_x %in% c("", "NaN", "NA", "None")) & !is.na(split_x)]

    # Return character(0) if split_x is empty after cleaning
    if (length(split_x) == 0) {
      return(character(0))
    } else {
      return(split_x)
    }
  })
}

# Apply the processing function to the columns
if (is_unix) {
  output_dt[, (array_columns) := mclapply(.SD, process_error_warning_column, mc.cores = parallel::detectCores()), .SDcols = array_columns]
} else {
  output_dt[, (array_columns) := lapply(.SD, process_error_warning_column), .SDcols = array_columns]
}

# Now 'output_dt' is your final result
# You can proceed to use 'output_dt' as needed

# Replace <NA> values in 'py_drg' and 'py_pdc' with character(0)
output_dt[, py_drg := ifelse(is.na(py_drg), "", py_drg)]
output_dt[, py_pdc := ifelse(is.na(py_pdc), "", py_pdc)]
# output_dt[, py_pccl := ifelse(is.nan(py_pccl), NA_real_, py_pccl)]

# For example, print the first few rows
# print(head(output_dt[id_series == 24465430]))
print(head(output_dt, 100))


     id_series py_drg py_pdc  py_pccl py_err py_warn
        <char> <char> <char>    <num> <list>  <list>
  1:  16908032  01670     1T      NaN               
  2:  21081178  15531             NaN               
  3:  31026539  13090   13PG      NaN               
  4:  44485625  26529    14A      NaN               
  5:  45231747  26509             NaN      2        
  6:   6435986  26529    14A      NaN               
  7:  48580584  11500    11A      NaN               
  8:  39787490  06640     6M      NaN               
  9:  21603327  28299   28QC      NaN               
 10:   3622659  14500    14A      NaN               
 11:  41908503  12510    12B      NaN               
 12:   6817449  26509             NaN      2        
 13:  46185908  10530    10C      NaN               
 14:  43101195  11580    11H      NaN               
 15:  48194375  01670     1T      NaN               
 16:  33956078  28299   28QC      NaN               
 17:  17181059  26509             NaN      2  

In [78]:
if (to_debug) fwrite(output_dt, "test3.csv")
str(output_dt)


Classes ‘data.table’ and 'data.frame':	502515 obs. of  6 variables:
 $ id_series: chr  "16908032" "21081178" "31026539" "44485625" ...
 $ py_drg   : chr  "01670" "15531" "13090" "26529" ...
 $ py_pdc   : chr  "1T" "" "13PG" "14A" ...
 $ py_pccl  : num  NaN NaN NaN NaN NaN NaN NaN NaN NaN NaN ...
 $ py_err   :List of 502515
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..

In [79]:
# Set the table name based on row count
bq_table <- if (nrow(output_dt) == total_rows) {
  paste0("python_", year_to_load)
} else {
  paste0("temp_python_", year_to_load)
}

# Check if the table should be dropped and replaced
if (to_drop_bq) {
  tryCatch(
    {
      bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
      message("Table dropped successfully.\n")
    },
    error = function(e) {
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.\n")
      } else {
        stop(e) # Re-throw other errors
      }
    }
  )
}

# Attempt to create the table
tryCatch(
  {
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here("data-cleaning/r_scripts", "bq_schema_python.json"), simplifyDataFrame = FALSE)
    )
    skip_bq_upload <<- FALSE
    message("Table created successfully.\n")
  },
  error = function(e) {
    if (grepl("already exists", e, ignore.case = TRUE)) {
      skip_bq_upload <<- TRUE
      message("Table already exists. Skipping creation and upload.\n")
    } else {
      stop(e) # Re-throw other errors
    }
  }
)

# Upload to BQ with batch logic if the table is empty
if (to_bq && !skip_bq_upload) {
  chunk_size <- 1000000 # Adjust chunk size based on memory availability
  num_chunks <- ceiling(nrow(output_dt) / chunk_size)

  message("Starting batch upload...\n")

  for (i in seq_len(num_chunks)) {
    chunk <- output_dt[((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(output_dt)), ]

    tryCatch(
      {
        bq_table_upload(
          bq_table(gcp_proj, bq_dataset, bq_table),
          values = chunk,
          write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
        )
        message(paste0("Chunk ", i, " uploaded successfully.\n"))
      },
      error = function(e) {
        if (grepl("already exists", e, ignore.case = TRUE)) {
          message("Upload skipped: table already exists and is not empty.\n")
          break # Stop further uploads
        } else {
          message(paste0("Error during upload for chunk ", i, ": ", e, "\n"))
        }
      }
    )
  }

  message("Batch upload completed.\n")
}


Auto-refreshing stale OAuth token.



## Run Thai Grouper

Run the Thai Grouper on your computer using the file from gs://phic-claims-checkpoints/pre-tdrg/

In [51]:
if (to_spc) {
  result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")))
} else {
  result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))
}
# str(result)

result_mapping <- result[, .(id_series, caseid = as.character(1:.N))]

export_for_grouper(
  result,
  here(checkpoint_4_path, paste0(
    checkpoint_4_prefix, year_to_load, suffix, ".txt"
  ))
)

# str(result)
# Upload the file
gcs_auth(email = gcs_email)
gcs_upload(
  file = here(checkpoint_4_path, paste0(checkpoint_4_prefix, year_to_load, suffix, ".txt")),
  bucket = gcs_bucket,
  name = paste0(gcs_pre_fpath, "/", paste0(checkpoint_4_prefix, year_to_load, suffix, ".txt")),
  predefinedAcl = "bucketLevel"
)

if (thai_prompt || to_prompt) {
  response <- tolower(readline(prompt = "Have you run the Thai grouper manually? (y/n): "))
  if (response == "y") {
    message("Continuing with the script...\n")
    # Continue with the rest of the script
  } else {
    message("Stopping the script.\n")
    stop("Thai Grouper not run yet. Script terminated. Continue on manually if necessary")
  }
} else {
  message("Thai Grouper is assumed to have been run already. Continuing with the script...\n")
}

gcs_get_object(
  object_name = paste0(gcs_post_fpath, "/", toupper(paste0(checkpoint_5_prefix, year_to_load, suffix)), "Res.TXT"),
  bucket = gcs_bucket,
  saveToDisk = here(checkpoint_5_path, paste0(toupper(paste0(checkpoint_5_prefix, year_to_load, suffix)), "Res.TXT")),
  overwrite = TRUE
)

thai_result <- fread(here(checkpoint_5_path, paste0(toupper(paste0(checkpoint_5_prefix, year_to_load, suffix)), "Res.TXT")), colClasses = "character")
if (to_debug) print(head(thai_result))
thai_result <- merge(
  thai_result,
  result_mapping[, .(caseid, id_series)], # Select only caseid and id_series from result_mapping
  by = "caseid",
  all.x = TRUE
)

thai_result[, caseid := id_series]
thai_result[, id_series := NULL]
thai_result[, thai_drg := drg]
thai_result[, thai_rw := rw]
thai_result[, thai_wtlos := wtlos]
thai_result[, thai_ot := ot]
thai_result[, thai_adjrw := adjrw]
thai_result[, thai_err := err]
thai_result[, thai_warn := warn]
thai_result[, thai_los := los]
thai_result[, drg := NULL]
thai_result[, drgname := NULL]
thai_result[, rw := NULL]
thai_result[, wtlos := NULL]
thai_result[, ot := NULL]
thai_result[, adjrw := NULL]
thai_result[, err := NULL]
thai_result[, warn := NULL]
thai_result[, los := NULL]

str(thai_result)


Rows where DOB will be updated from recomputed values (Before):
Key: <CASEID>
Empty data.table (0 rows and 2 cols): CASEID,DOB_before
Rows where DOB was updated from recomputed values (After):
Key: <CASEID>
Empty data.table (0 rows and 2 cols): CASEID,DOB_after
Classes ‘data.table’ and 'data.frame':	184399 obs. of  42 variables:
 $ CASEID : chr  "1" "10" "100" "1000" ...
 $ DOB    : chr  "--" "--" "--" "--" ...
 $ Sex    : num  2 1 2 2 1 1 2 1 2 2 ...
 $ DateAdm: chr  "09/02/2022" "02/09/2022" "17/02/2022" "12/01/2022" ...
 $ TimeAdm: chr  "2020" "2130" "1540" "0818" ...
 $ DateDsc: chr  "16/02/2022" "12/09/2022" "19/02/2022" "14/01/2022" ...
 $ TimeDsc: chr  "1545" "1208" "1400" "1135" ...
 $ DischT : chr  "1" "1" "1" "1" ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "K319" "S2240" "O800" "O821" ...
 $ SDx1   : chr  "--" "K291" "Z370" "O321" ...
 $ SDx2   : chr  "--" "S271" "--" "O649" ...
 $ SDx3   : chr  "--" "S42002" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--

ℹ 2024-10-16 09:01:17.355772 > File size detected as  26.7 Mb

ℹ 2024-10-16 09:01:17.422275 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/phic-claims-checkpoints/o/?uploadType=resumable&name=pre-tdrg%2Fcheckpoint_4_thai_grouper_input_2022_sampled_34019_.txt&upload_id=AHmUCY08DCdjfWbg0NKAwIUwWUjZR6GYZjLnOl4eGFAyJlnqAe3yy28JVHH9BvJ3pVPeIIyT3YXXACt0-ZYFUf9YsRJKCVg4269b3X5IiFQiZFx4xQ



Continuing with the script...


ℹ Downloading post-tdrg/PRE-TDRG_CHECKPOINT_4_THAI_GROUPER_INPUT_2022_SAMPLED_3…

✔ Saved post-tdrg/PRE-TDRG_CHECKPOINT_4_THAI_GROUPER_INPUT_2022_SAMPLED_34019_R…





[1] TRUE

Classes ‘data.table’ and 'data.frame':	184399 obs. of  9 variables:
 $ caseid    : chr  "3006563006013302206003" "8106748106010212206018" "0006930006061103206000" "6006856006061602206006" ...
 $ thai_drg  : chr  "26539" "26539" "26539" "26539" ...
 $ thai_rw   : chr  "0.0000" "0.0000" "0.0000" "0.0000" ...
 $ thai_wtlos: chr  "0.00" "0.00" "0.00" "0.00" ...
 $ thai_ot   : chr  "0" "0" "0" "0" ...
 $ thai_adjrw: chr  "0.0000" "0.0000" "0.0000" "0.0000" ...
 $ thai_err  : chr  "6" "6" "6" "6" ...
 $ thai_warn : chr  "9" "9" "11" "11" ...
 $ thai_los  : chr  "7" "10" "2" "2" ...
 - attr(*, ".internal.selfref")=<externalptr> 


In [52]:
if (exists("output_dt")) {
  before_merge <- data.table::copy(output_dt)
  str(before_merge)
  before_merge[, caseid := id_series]

  if (to_debug) print(head(before_merge))
  merged <- merge(before_merge, thai_result, by = "caseid", all.x = TRUE)
  if (to_debug) print(head(merged))
  diff_merged <- merged[!as.character(ifelse(is.na(py_drg), "NA", py_drg)) == as.character(thai_drg)]
  print(nrow(diff_merged))
  fwrite(diff_merged, here(checkpoint_9_path, paste0("checkpoint_9_grouper_differences_", year_to_load, suffix, ".csv")))
}


Classes ‘data.table’ and 'data.frame':	510285 obs. of  6 variables:
 $ id_series: chr  NA NA NA NA ...
 $ py_drg   : chr  "14500" "15530" "26549" "14500" ...
 $ py_pdc   : chr  "14A" "" "" "14A" ...
 $ py_pccl  : num  NaN NaN NaN NaN NaN NaN NaN NaN NaN NaN ...
 $ py_err   :List of 510285
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr "4"
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "4"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr

In [53]:
if (exists("merged")) str(merged)
if (to_debug) fwrite(merged, "test4.csv")


Classes ‘data.table’ and 'data.frame':	510285 obs. of  15 variables:
 $ caseid    : chr  NA NA NA NA ...
 $ id_series : chr  NA NA NA NA ...
 $ py_drg    : chr  "14500" "15530" "26549" "14500" ...
 $ py_pdc    : chr  "14A" "" "" "14A" ...
 $ py_pccl   : num  NaN NaN NaN NaN NaN NaN NaN NaN NaN NaN ...
 $ py_err    :List of 510285
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr "4"
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "4"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "2"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr 
  ..$ : chr "9"
  ..$ : chr 
  ..$ : 

Format data for BQ push

In [54]:
# Please run thai grouper first
result_after_thai <- data.table::copy(thai_result)

# Convert data types to match BigQuery schema
# format as full numbers, no exponential form
result_after_thai[, id_series := caseid]
result_after_thai[, caseid := NULL]
# format as full numbers, no exponential form
# result_after_thai[, id_pin := trimws(formatC(as.integer(id_pin), format = "f", digits = 0))]
# result_after_thai[, date_adm := as.Date(date_adm, format = "%Y-%m-%d")]
# result_after_thai[, time_adm := as.ITime(time_adm)]
# result_after_thai[, date_dis := as.Date(date_dis, format = "%Y-%m-%d")]
# result_after_thai[, time_dis := as.ITime(time_dis)]
# result_after_thai[, date_rec := as.Date(date_rec, format = "%Y-%m-%d")]
# result_after_thai[, date_ref := as.Date(date_ref, format = "%Y-%m-%d")]
# result_after_thai[, date_check := as.Date(date_check, format = "%Y-%m-%d")]
# result_after_thai[, id_hci := as.character(id_hci)]
# result_after_thai[, id_hcp := id_hcp] # as is

# # Convert character "0"/"1" to logical for Boolean fields
# result_after_thai[, clin_outpatient := as.logical(clin_outpatient)] # as is
# result_after_thai[, clin_emergency := as.logical(clin_emergency)] # as is

# result_after_thai[, pat_type := as.character(pat_type)]
# result_after_thai[, clin_acc := as.character(clin_acc)]
# result_after_thai[, pat_rel := as.character(pat_rel)]
# result_after_thai[, pat_bdate := as.Date(pat_bdate, format = "%Y-%m-%d")]
# result_after_thai[, pat_age := as.numeric(pat_age)]
# result_after_thai[, pat_sex := as.character(pat_sex)]
# result_after_thai[, pat_bwt := as.numeric(pat_bwt)]
# result_after_thai[, pat_memcat_parent := as.character(pat_memcat_parent)]
# result_after_thai[, pat_memcat_child := as.character(pat_memcat_child)]
# result_after_thai[, clin_discharge := as.integer(clin_discharge)]
# # result[, clin_c1 := clin_c1]
# # result[, clin_c2 := clin_c2]

# result_after_thai[, claim_status := as.character(claim_status)]
# result_after_thai[, claim_payout := as.numeric(claim_payout)]
# result_after_thai[, claim_charge := as.numeric(claim_charge)]
# result_after_thai[, date_ext := as.Date(date_ext, format = "%Y-%m-%d")]
# result_after_thai[, id_year := as.integer(id_year)]

# if (is_unix) {
#   result_after_thai[, clin_sdx := mclapply(clin_sdx, function(x) if (all(is.na(x))) character(0) else x)]
# } else {
#   result_after_thai[, clin_sdx := future_lapply(clin_sdx, function(x) if (all(is.na(x))) character(0) else x)]
# }

# result_after_thai[, clin_proc := clin_rvs] # as is
# if (is_unix) {
#   result_after_thai[, clin_proc := mclapply(clin_proc, function(x) if (all(is.na(x))) character(0) else x)]
# } else {
#   result_after_thai[, clin_proc := future_lapply(clin_proc, function(x) if (all(is.na(x))) character(0) else x)]
# }

# result_after_thai[, clin_rvs := NULL] # as is
# result_after_thai[, pat_ageday := as.integer(ageday)] # as is
# result_after_thai[, ageday := NULL]

# result_after_thai[, clin_pdx := as.character(clin_pdx)]
# result_after_thai[, clin_pdx_source := as.integer(pdx_code)]
# result_after_thai[, pdx_code := NULL]

result_after_thai[, thai_drg := as.character(thai_drg)]
result_after_thai[, thai_rw := as.numeric(thai_rw)]
result_after_thai[, thai_wtlos := as.numeric(thai_wtlos)]
result_after_thai[, thai_ot := as.integer(thai_ot)]
result_after_thai[, thai_adjrw := as.numeric(thai_adjrw)]
result_after_thai[, thai_err := as.integer(thai_err)]
result_after_thai[, thai_warn := as.integer(thai_warn)]
result_after_thai[, thai_los := as.integer(thai_los)]

# result_after_thai[, py_pdc := as.character(py_pdc)]
# result_after_thai[, py_pccl := as.numeric(py_pccl)]
# result_after_thai[, py_drg := as.character(py_drg)]
# result_after_thai[, py_warn := py_warn] # as is
# result_after_thai[, py_err := py_err] # as is

# Reorder the columns in the result data.table to match the schema
setcolorder(result_after_thai, c(
  # "caseid",
  # "id_year",
  "id_series",
  # "id_pin",
  # "id_hci",
  # "id_hcp",
  # "date_adm",
  # "time_adm",
  # "date_dis",
  # "time_dis",
  # "date_rec",
  # "date_ref",
  # "date_check",
  # "date_ext",
  # "pat_type",
  # "pat_rel",
  # "pat_bdate",
  # "pat_age",
  # "pat_ageday",
  # "pat_sex",
  # "pat_bwt",
  # "pat_memcat_parent",
  # "pat_memcat_child",
  # "claim_status",
  # "claim_payout",
  # "claim_charge",
  # "clin_discharge",
  # "clin_outpatient",
  # "clin_emergency",
  # "clin_acc",
  # "clin_c1",
  # "clin_c2",
  # "clin_sdx",
  # "clin_proc",
  # "clin_pdx",
  # "clin_pdx_source",
  "thai_drg",
  "thai_rw",
  "thai_wtlos",
  "thai_ot",
  "thai_adjrw",
  "thai_err",
  "thai_warn",
  "thai_los" # ,
  # "py_drg",
  # "py_pdc",
  # "py_pccl",
  # "py_warn",
  # "py_err"
))

# result_after_thai[, icd9_list := NULL]
# result_after_thai[, pat_age_orig := NULL]

# result_after_thai[, c1_orig := NULL]
# result_after_thai[, c2_orig := NULL]
# result_after_thai[, c1 := NULL]
# result_after_thai[, c2 := NULL]


Check if any of our identifiers are in exponential for, e.g. "1.9e+07"

In [55]:
# str(result_after_thai)
print(result_after_thai[grepl("e", id_series)])
# print(result_after_thai[grepl("e", id_pin)])
# print(result_after_thai[grepl("e", id_hci)])


Empty data.table (0 rows and 9 cols): id_series,thai_drg,thai_rw,thai_wtlos,thai_ot,thai_adjrw...


In [56]:
str(result_after_thai)


Classes ‘data.table’ and 'data.frame':	184399 obs. of  9 variables:
 $ id_series : chr  "3006563006013302206003" "8106748106010212206018" "0006930006061103206000" "6006856006061602206006" ...
 $ thai_drg  : chr  "26539" "26539" "26539" "26539" ...
 $ thai_rw   : num  0 0 0 0 0 0 0 0 0 0 ...
 $ thai_wtlos: num  0 0 0 0 0 0 0 0 0 0 ...
 $ thai_ot   : int  0 0 0 0 0 0 0 0 0 0 ...
 $ thai_adjrw: num  0 0 0 0 0 0 0 0 0 0 ...
 $ thai_err  : int  6 6 6 6 6 6 6 6 6 6 ...
 $ thai_warn : int  9 9 11 11 9 9 11 9 11 9 ...
 $ thai_los  : int  7 10 2 2 2 4 2 15 8 4 ...
 - attr(*, ".internal.selfref")=<externalptr> 


## Push to BQ

Push results to BQ if appropriate (i.e. if allowed by parameters)

In [57]:
# Set the table name based on row count
bq_table <- if (nrow(result_after_thai) == total_rows) {
  paste0("thai_", year_to_load)
} else {
  paste0("temp_thai_", year_to_load)
}

# Check if the table should be dropped and replaced
if (to_drop_bq) {
  tryCatch(
    {
      bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
      message("Table dropped successfully.\n")
    },
    error = function(e) {
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.\n")
      } else {
        stop(e) # Re-throw other errors
      }
    }
  )
}

# Attempt to create the table
tryCatch(
  {
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here("data-cleaning/r_scripts", "bq_schema_thai.json"), simplifyDataFrame = FALSE)
    )
    skip_bq_upload <<- FALSE
    message("Table created successfully.\n")
  },
  error = function(e) {
    if (grepl("already exists", e, ignore.case = TRUE)) {
      skip_bq_upload <<- TRUE
      message("Table already exists. Skipping creation and upload.\n")
    } else {
      stop(e) # Re-throw other errors
    }
  }
)

# Upload to BQ only if the table is empty
if (to_bq && !skip_bq_upload) {
  chunk_size <- 1000000 # Adjust chunk size based on memory availability
  num_chunks <- ceiling(nrow(result_after_thai) / chunk_size)

  message("Starting batch upload...\n")

  for (i in seq_len(num_chunks)) {
    chunk <- result_after_thai[((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result_after_thai)), ]

    tryCatch(
      {
        bq_table_upload(
          bq_table(gcp_proj, bq_dataset, bq_table),
          values = chunk,
          write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
        )
        message(paste0("Chunk ", i, " uploaded successfully.\n"))
      },
      error = function(e) {
        if (grepl("already exists", e, ignore.case = TRUE)) {
          message("Upload skipped: table already exists and is not empty.\n")
          break # Stop further uploads
        } else {
          message(paste0("Error during upload for chunk ", i, ": ", e, "\n"))
        }
      }
    )
  }

  message("Batch upload completed.\n")
}


Table does not exist, nothing to drop.




Table created successfully.


Starting batch upload...


Chunk 1 uploaded successfully.


Batch upload completed.




In [58]:
saveRDS(result_after_thai, here(checkpoint_6_path, paste0(checkpoint_6_prefix, year_to_load, suffix, ".rds")))
